# Notebook 06 – Guideline Validation and Scoring

Combine deterministic evidence from Notebooks 03, 04, and 05 into a single **Guideline Confidence Score**.

No LLM — pure weighted scoring on normalized metrics.

| Source | Metrics |
|--------|---------|
| Notebook 03 | Cramér's V, statistical evidence |
| Notebook 04 | Random Forest importance, mean \|SHAP\| |
| Notebook 05 | Support, confidence, lift |

Records are merged on **Predictor × UI Element**. Only **Strong** and **Very Strong** guidelines are exported for Notebook 07.


## Expected Outputs
- `reports/Guideline_Validation/validated_guidelines.csv` — top 100 verified guidelines
- `reports/Guideline_Validation/guideline_scores.xlsx` — full scores + strength summary
- `reports/Guideline_Validation/guideline_summary.md`


In [11]:
import logging
import sys
from pathlib import Path

import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate
    raise FileNotFoundError("Could not find project root containing src/config.py.")


PROJECT_ROOT = _bootstrap_project()

from src.evidence_engine.pipeline import run_guideline_scoring_pipeline
from src.evidence_engine.scoring import GUIDELINE_SCORE_WEIGHTS
from src.utils.notebook import setup_notebook

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PATHS, REPORTS = setup_notebook("Guideline_Validation")
TOP_N = 100

print("Guideline Confidence Score weights:")
for component, weight in GUIDELINE_SCORE_WEIGHTS.items():
    print(f"  {component.replace('_', ' ')}: {weight:.0%}")


Guideline Confidence Score weights:
  Statistical Evidence: 25%
  Feature Importance: 25%
  SHAP: 20%
  Confidence: 15%
  Lift: 15%


## Load Upstream Evidence

Requires completed runs of Notebooks 03–05.


In [12]:
STAT_PATH = PATHS.reports / "Statistical_Validation" / "tables" / "statistical_results.csv"
IMPORTANCE_PATH = PATHS.reports / "Feature_Importance" / "feature_importance.csv"
SHAP_PATH = PATHS.reports / "Feature_Importance" / "shap_summary.csv"
RULES_PATH = PATHS.reports / "AssociationRules" / "association_rules.csv"

for label, path in [
    ("Statistical (NB03)", STAT_PATH),
    ("Feature importance (NB04)", IMPORTANCE_PATH),
    ("SHAP (NB04)", SHAP_PATH),
    ("Association rules (NB05)", RULES_PATH),
]:
    status = "OK" if path.exists() else "MISSING — run upstream notebook first"
    print(f"{label}: {status} ({path.name})")


Statistical (NB03): OK (statistical_results.csv)
Feature importance (NB04): OK (feature_importance.csv)
SHAP (NB04): OK (shap_summary.csv)
Association rules (NB05): OK (association_rules.csv)


## Score and Export Guidelines


In [13]:
result = run_guideline_scoring_pipeline(
    PROJECT_ROOT,
    REPORTS,
    top_n=TOP_N,
)

all_scores = result.all_scores
validated = result.validated_guidelines

print(f"Total pairs scored: {len(all_scores)}")
print(f"Validated (Strong + Very Strong): {all_scores['Is_Validated'].sum()}")
print(f"Top {TOP_N} exported: {len(validated)}")
print("\nStrength distribution:")
print(all_scores["Guideline_Strength"].value_counts().to_string())


INFO: Exported guideline scoring outputs to /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Guideline_Validation


Total pairs scored: 328
Validated (Strong + Very Strong): 126
Top 100 exported: 100

Strength distribution:
Guideline_Strength
Moderate       174
Strong         121
Weak            28
Very Strong      5


## Top Validated Guidelines


In [14]:
display_cols = [
    "Predictor",
    "UI_Element",
    "Guideline_Confidence_Score",
    "Guideline_Strength",
    "Cramers_V",
    "RF_Importance",
    "Mean_SHAP",
    "Rule_Confidence",
    "Rule_Lift",
    "Norm_Cramers_V",
    "Norm_RF_Importance",
    "Norm_Mean_SHAP",
    "Norm_Confidence",
    "Norm_Lift",
]
available = [col for col in display_cols if col in validated.columns]
display(validated[available].head(20))


,Predictor,UI_Element,Guideline_Confidence_Score,Guideline_Strength,Cramers_V,RF_Importance,Mean_SHAP,Rule_Confidence,Rule_Lift,Norm_Cramers_V,Norm_RF_Importance,Norm_Mean_SHAP,Norm_Confidence,Norm_Lift
0,Agreeableness_Level,mobile_sticky_header,0.783911,Very Strong,0.150667,0.148851,0.078129,0.785714,1.289075,0.587979,0.823773,1.000000,0.785714,0.754109
1,primary_persona,mobile_sticky_header,0.741986,Very Strong,0.224155,0.144839,0.031087,0.875000,1.356589,0.874764,0.797112,0.368633,0.875000,0.793605
2,current_mood,desktop_persistent_filters,0.732949,Very Strong,0.240211,0.146129,0.026984,0.875000,1.176194,0.937424,0.805682,0.313556,0.875000,0.688073
3,Openness_Level,mobile_category_display,0.731728,Very Strong,0.185356,0.172982,0.030042,0.656250,1.544118,0.723354,0.984139,0.354604,0.656250,0.903309
4,Agreeableness_Level,mobile_review_display,0.701535,Very Strong,0.169369,0.137327,0.045867,0.725000,1.451247,0.660962,0.747190,0.567003,0.725000,0.848980
5,Conscientiousness_Level,mobile_whitespace,0.698410,Strong,0.129052,0.175369,0.022015,0.967742,1.458333,0.503627,1.000000,0.246865,0.967742,0.853125
6,primary_persona,mobile_price_display,0.694557,Strong,0.217061,0.123170,0.032497,0.833333,1.333333,0.847079,0.653108,0.387553,0.833333,0.780000
7,current_mood,desktop_image_text_ratio,0.693115,Strong,0.236792,0.139843,0.023634,0.692308,1.294033,0.924079,0.763909,0.268601,0.692308,0.757009
8,current_mood,mobile_filter_location,0.685664,Strong,0.233593,0.144385,0.006663,0.833333,1.436782,0.911597,0.794096,0.040815,0.833333,0.840517
9,Neuroticism_Level,desktop_persistent_filters,0.683598,Strong,0.161669,0.167021,0.020742,0.913043,1.217391,0.630915,0.944525,0.229779,0.913043,0.712174


## Export Locations


In [15]:
print("Exports:")
for name, path in result.export_paths.items():
    print(f"- {name}: {path}")

if not validated.empty:
    top = validated.iloc[0]
    print(
        f"\nTop guideline: {top['Predictor']} → {top['UI_Element']} "
        f"(score={top['Guideline_Confidence_Score']:.3f}, {top['Guideline_Strength']})"
    )

print("\nReady for Notebook 07 — Guideline JSON.")


Exports:
- validated_guidelines_csv: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Guideline_Validation/validated_guidelines.csv
- guideline_scores_xlsx: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Guideline_Validation/guideline_scores.xlsx
- guideline_summary_md: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Guideline_Validation/guideline_summary.md

Top guideline: Agreeableness_Level → mobile_sticky_header (score=0.784, Very Strong)

Ready for Notebook 07 — Guideline JSON.
